# Grad-CAM Analyse
Baseline vs. Augmentation – Testdatensatz
Vgl. Kapitel 4.5 und 5.4

In [ ]:
import sys
sys.path.append('..')

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from config import DATA, PATHS
from data.loader import load_test_dataset
from evaluation.grad_cam import (
    compute_grad_cam,
    overlay_heatmap,
    classify_test_images,
    plot_grad_cam_row
)
from training.train_augmentation import build_augmentation

CLASS_NAMES = DATA['classes']
print('Setup abgeschlossen.')

## Zelle 2: Modelle + Testdatensatz laden

In [ ]:
# Beide Keras-Modelle laden
baseline    = tf.keras.models.load_model(str(PATHS['models'] / 'baseline.keras'))
augmentation = tf.keras.models.load_model(str(PATHS['models'] / 'augmentation_rotation_brightness.keras'))

print('Baseline geladen:     ', str(PATHS['models'] / 'baseline.keras'))
print('Augmentation geladen: ', str(PATHS['models'] / 'augmentation_rotation_brightness.keras'))

# Testdatensatz laden (batch_size=1, shuffle=False)
test_ds = load_test_dataset(batch_size=1)
print(f'Testbilder: {len(list(test_ds))}')

In [ ]:
from evaluation.grad_cam import LAYER_NAME

base_model = baseline.get_layer('mobilenetv2_1.00_224')
layer = base_model.get_layer(LAYER_NAME)

print(f"Verwendete Grad-CAM Schicht : {LAYER_NAME}")
print(f"Layer-Typ                   : {layer.__class__.__name__}")
print(f"Output-Shape                : {layer.output_shape}")

## Zelle 3: Alle Testbilder klassifizieren

In [ ]:
# Klassifikation für beide Modelle
print('Klassifiziere mit Baseline...')
images_list, y_true_b, y_pred_b, conf_b = classify_test_images(baseline, test_ds)

print('Klassifiziere mit Augmentation...')
_, y_true_a, y_pred_a, conf_a = classify_test_images(augmentation, test_ds)

# Übersicht
print(f'\nBaseline    – Accuracy: {np.mean(y_true_b == y_pred_b):.4f}')
print(f'Augmentation – Accuracy: {np.mean(y_true_a == y_pred_a):.4f}')

# Auswahlkategorien berechnen
wrong_b    = np.where(y_true_b != y_pred_b)[0]
wrong_a    = np.where(y_true_a != y_pred_a)[0]
low_conf_b = np.argsort(conf_b)[:5]
low_conf_a = np.argsort(conf_a)[:5]

print(f'\nFehlklassifikationen Baseline:     {len(wrong_b)}')
print(f'Fehlklassifikationen Augmentation: {len(wrong_a)}')

## Zelle 4: Korrekte Klassifikationen – Baseline vs. Augmentation
2 Bilder je Klasse, beide Modelle nebeneinander

In [ ]:
baseline.summary()

In [ ]:
for class_idx, class_name in enumerate(CLASS_NAMES):
    # 2 korrekte Bilder dieser Klasse
    correct_idx = np.where(
        (y_true_b == class_idx) & (y_pred_b == class_idx)
    )[0][:2]

    if len(correct_idx) == 0:
        print(f'Keine korrekten Klassifikationen für {class_name}')
        continue

    for idx in correct_idx:
        image = images_list[idx]

        # Grad-CAM für beide Modelle
        heatmap_b = compute_grad_cam(baseline,     image, class_idx)
        heatmap_a = compute_grad_cam(augmentation, image, class_idx)

        fig, axes = plt.subplots(2, 3, figsize=(14, 8))
        fig.suptitle(
            f'Korrekte Klassifikation – Klasse: {class_name} (Index {idx})',
            fontsize=13
        )

        # Zeile 1: Baseline
        axes[0, 0].set_ylabel('Baseline', fontsize=11)
        plot_grad_cam_row(
            axes[0],
            image, heatmap_b,
            CLASS_NAMES[y_true_b[idx]],
            CLASS_NAMES[y_pred_b[idx]],
            conf_b[idx]
        )

        # Zeile 2: Augmentation
        axes[1, 0].set_ylabel('Augmentation', fontsize=11)
        plot_grad_cam_row(
            axes[1],
            image, heatmap_a,
            CLASS_NAMES[y_true_a[idx]],
            CLASS_NAMES[y_pred_a[idx]],
            conf_a[idx]
        )

        plt.tight_layout()
        output_path = PATHS['grad_cam'] / f'correct_{class_name}_{idx}.png'
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Gespeichert: {output_path}')


        from evaluation.grad_cam import LAYER_NAME


## Zelle 5: Fehlklassifikationen – alle
Baseline und Augmentation getrennt

In [ ]:
for model_name, wrong, y_true, y_pred, conf, model in [
    ('baseline',     wrong_b, y_true_b, y_pred_b, conf_b, baseline),
    ('augmentation', wrong_a, y_true_a, y_pred_a, conf_a, augmentation)
]:
    if len(wrong) == 0:
        print(f'{model_name}: keine Fehlklassifikationen')
        continue

    print(f'\n{model_name} – {len(wrong)} Fehlklassifikationen:')

    fig, axes = plt.subplots(len(wrong), 3, figsize=(12, 4 * len(wrong)))
    if len(wrong) == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(f'Fehlklassifikationen – {model_name}', fontsize=13)

    for row, idx in enumerate(wrong):
        image   = images_list[idx]
        heatmap = compute_grad_cam(model, image, y_pred[idx])

        plot_grad_cam_row(
            axes[row],
            image, heatmap,
            CLASS_NAMES[y_true[idx]],
            CLASS_NAMES[y_pred[idx]],
            conf[idx]
        )
        print(f'  Index {idx}: Wahr={CLASS_NAMES[y_true[idx]]} '
              f'Pred={CLASS_NAMES[y_pred[idx]]} '
              f'Konfidenz={conf[idx]:.4f}')

    plt.tight_layout()
    output_path = PATHS['grad_cam'] / f'misclassified_{model_name}.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {output_path}')

In [ ]:
# Zelle im Notebook ergänzen:
augment = build_augmentation()

# Augmentierte Varianten eines Testbildes erzeugen
augmented_variants = [
    ('Original',         image),
    ('RandomFlip',       tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal_and_vertical')])(tf.expand_dims(image, 0), training=True)[0]),
    ('RandomRotation',   tf.keras.Sequential([tf.keras.layers.RandomRotation(0.1)])(tf.expand_dims(image, 0), training=True)[0]),
    ('RandomBrightness', tf.keras.Sequential([tf.keras.layers.RandomBrightness(0.2)])(tf.expand_dims(image, 0), training=True)[0]),
    ('RandomZoom',       tf.keras.Sequential([tf.keras.layers.RandomZoom((-0.2, 0.0))])(tf.expand_dims(image, 0), training=True)[0]),
]

# Grad-CAM für jede Variante mit beiden Modellen
for transform_name, img in augmented_variants:
    heatmap_b = compute_grad_cam(baseline,     img, predicted_class)
    heatmap_a = compute_grad_cam(augmentation, img, predicted_class)
    # → plot

## Zelle 6: Grenzfälle – niedrige Konfidenz

In [ ]:
for model_name, low_conf_idx, y_true, y_pred, conf, model in [
    ('baseline',     low_conf_b, y_true_b, y_pred_b, conf_b, baseline),
    ('augmentation', low_conf_a, y_true_a, y_pred_a, conf_a, augmentation)
]:
    print(f'\n{model_name} – Grenzfälle (niedrigste Konfidenz):')

    fig, axes = plt.subplots(len(low_conf_idx), 3, figsize=(12, 4 * len(low_conf_idx)))
    if len(low_conf_idx) == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(f'Grenzfälle – {model_name}', fontsize=13)

    for row, idx in enumerate(low_conf_idx):
        image   = images_list[idx]
        heatmap = compute_grad_cam(model, image, y_pred[idx])

        plot_grad_cam_row(
            axes[row],
            image, heatmap,
            CLASS_NAMES[y_true[idx]],
            CLASS_NAMES[y_pred[idx]],
            conf[idx]
        )
        print(f'  Index {idx}: Wahr={CLASS_NAMES[y_true[idx]]} '
              f'Pred={CLASS_NAMES[y_pred[idx]]} '
              f'Konfidenz={conf[idx]:.4f}')

    plt.tight_layout()
    output_path = PATHS['grad_cam'] / f'low_confidence_{model_name}.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {output_path}')

## Zelle 7: Direkter Vergleich – dasselbe Bild durch beide Modelle
Wähle einen Index manuell aus den obigen Ergebnissen

In [ ]:
# ── Index hier manuell setzen ─────────────────────────────
IDX = 0    # anpassen nach Analyse der obigen Zellen
# ─────────────────────────────────────────────────────────

image = images_list[IDX]

heatmap_b = compute_grad_cam(baseline,     image, y_pred_b[IDX])
heatmap_a = compute_grad_cam(augmentation, image, y_pred_a[IDX])

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(
    f'Direkter Vergleich – Index {IDX} | '
    f'Wahr: {CLASS_NAMES[y_true_b[IDX]]}',
    fontsize=13
)

axes[0, 0].set_ylabel('Baseline', fontsize=11)
plot_grad_cam_row(
    axes[0], image, heatmap_b,
    CLASS_NAMES[y_true_b[IDX]],
    CLASS_NAMES[y_pred_b[IDX]],
    conf_b[IDX]
)

axes[1, 0].set_ylabel('Augmentation', fontsize=11)
plot_grad_cam_row(
    axes[1], image, heatmap_a,
    CLASS_NAMES[y_true_a[IDX]],
    CLASS_NAMES[y_pred_a[IDX]],
    conf_a[IDX]
)

plt.tight_layout()
output_path = PATHS['grad_cam'] / f'comparison_idx{IDX}.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Gespeichert: {output_path}')